In [ ]:
# ============================================================
#  MPHOnline.com — Full BeautifulSoup Scraper
#  Platform : Google Colab
#  Target   : https://mphonline.com  (Shopify store)
#  Potential: 55 000+ book records
# ============================================================

# ── CELL 1 · Install dependencies ───────────────────────────
# Run this cell first in Colab
!pip install requests beautifulsoup4 lxml pandas tqdm -q

In [ ]:
# ── CELL 2 · Imports & configuration ────────────────────────
import requests
import time
import random
import json
import re
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
from pathlib import Path

# ── Politeness settings (important — do NOT remove) ─────────
MIN_DELAY   = 1.5   # seconds between requests (min)
MAX_DELAY   = 3.5   # seconds between requests (max)
MAX_RETRIES = 3

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

BASE_URL = "https://mphonline.com"

OUTPUT_DIR = Path("mph_data")
OUTPUT_DIR.mkdir(exist_ok=True)

print("✅ Setup complete.")


✅ Setup complete.


In [ ]:
# ── CELL 3 · Helper: safe HTTP GET ──────────────────────────
def get_page(url: str, params: dict = None) -> BeautifulSoup | None:
    """
    Fetch a URL with retry logic and polite delays.
    Returns a BeautifulSoup object or None on failure.
    """
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))
            resp = requests.get(url, headers=HEADERS, params=params, timeout=15)
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            elif resp.status_code == 429:
                wait = 30 * attempt
                print(f"  ⚠️  Rate limited. Waiting {wait}s …")
                time.sleep(wait)
            else:
                print(f"  ✗ HTTP {resp.status_code} for {url}")
                return None
        except requests.RequestException as e:
            print(f"  ✗ Request error (attempt {attempt}): {e}")
            time.sleep(5 * attempt)
    return None


In [ ]:
# ── CELL 4 · Operation 1 · Scrape ALL collection slugs ──────
"""
OPERATION 1 — Discover all collections
MPHOnline is Shopify. Every collection is at:
  /collections/<slug>
We harvest slugs from the site-nav links on the homepage.
"""

def scrape_all_collections() -> list[dict]:
    soup = get_page(BASE_URL)
    if not soup:
        return []

    seen = set()
    collections = []

    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/collections/" in href:
            # Normalise
            slug = href.split("/collections/")[-1].split("?")[0].rstrip("/")
            if slug and slug not in seen:
                seen.add(slug)
                name = a.get_text(strip=True) or slug
                collections.append({
                    "slug": slug,
                    "name": name,
                    "url": f"{BASE_URL}/collections/{slug}"
                })

    df = pd.DataFrame(collections)
    df.to_csv(OUTPUT_DIR / "collections.csv", index=False)
    print(f"✅ Op 1 — Found {len(collections)} collections → mph_data/collections.csv")
    return collections

collections = scrape_all_collections()

✅ Op 1 — Found 359 collections → mph_data/collections.csv


In [ ]:
# ── CELL 5 · Operation 2 · Shopify JSON API — full product list ─
"""
OPERATION 2 — Bulk product list via Shopify's public JSON API
Every public Shopify store exposes:
  /products.json?limit=250&page=N
This is the fastest way to reach 55 000+ records.
Fields: id, title, handle, vendor, product_type, tags,
        published_at, variants (price, sku, availability),
        images, body_html (description)
"""

def scrape_products_json(max_pages: int = 300) -> pd.DataFrame:
    all_products = []
    url = f"{BASE_URL}/products.json"

    for page in tqdm(range(1, max_pages + 1), desc="Fetching product pages"):
        time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))
        try:
            resp = requests.get(
                url,
                headers=HEADERS,
                params={"limit": 250, "page": page},
                timeout=15
            )
            if resp.status_code == 200:
                data = resp.json()
                products = data.get("products", [])
                if not products:
                    print(f"\n  ⏹ No more products at page {page}. Done.")
                    break
                all_products.extend(products)
                print(f"  Page {page}: +{len(products)} | Total: {len(all_products)}")
            elif resp.status_code == 429:
                print(f"\n  ⚠️  Rate limited on page {page}. Sleeping 60s …")
                time.sleep(60)
            else:
                print(f"\n  ✗ HTTP {resp.status_code} on page {page}. Stopping.")
                break
        except Exception as e:
            print(f"\n  ✗ Error on page {page}: {e}")
            time.sleep(10)

    rows = []
    for p in all_products:
        # Flatten variants (price tiers, ISBNs stored as SKUs)
        for v in p.get("variants", [{}]):
            rows.append({
                "product_id"      : p.get("id"),
                "title"           : p.get("title"),
                "handle"          : p.get("handle"),
                "vendor"          : p.get("vendor"),        # Publisher
                "product_type"    : p.get("product_type"),  # Category
                "tags"            : ", ".join(p.get("tags", [])),
                "published_at"    : p.get("published_at"),
                "description_html": p.get("body_html"),
                "variant_id"      : v.get("id"),
                "sku"             : v.get("sku"),           # Often ISBN
                "price_myr"       : v.get("price"),
                "compare_at_price": v.get("compare_at_price"),
                "available"       : v.get("available"),
                "variant_title"   : v.get("title"),
                "image_url"       : (p.get("images") or [{}])[0].get("src"),
                "product_url"     : f"{BASE_URL}/products/{p.get('handle')}",
            })

    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_DIR / "products_full.csv", index=False)
    print(f"\n✅ Op 2 — {len(df)} rows saved → mph_data/products_full.csv")
    return df

df_products = scrape_products_json()
df_products.head(3)


Fetching product pages:   0%|          | 1/300 [00:03<16:32,  3.32s/it]

  Page 1: +250 | Total: 250


Fetching product pages:   1%|          | 2/300 [00:06<16:43,  3.37s/it]

  Page 2: +250 | Total: 500


Fetching product pages:   1%|          | 3/300 [00:10<17:03,  3.44s/it]

...

  Page 100: +250 | Total: 25000


Fetching product pages:  33%|███▎      | 100/300 [05:45<11:31,  3.46s/it]


  ✗ HTTP 400 on page 101. Stopping.



✅ Op 2 — 25000 rows saved → mph_data/products_full.csv


,product_id,title,handle,vendor,product_type,tags,published_at,description_html,variant_id,sku,price_myr,compare_at_price,available,variant_title,image_url,product_url
0,8922744815773,THE MYSTERY OF THE VANISHED PRINCE #9,the-mystery-of-the-vanished-prince-9,MPHOnline.com,,"Class Code_226, Class_TWEENS (YOUNG READER) (9...",2026-05-17T19:00:05+08:00,None,48302515519645,MYSTERIES9,12.90,0.00,False,Default Title,None,https://mphonline.com/products/the-mystery-of-...
1,8922744783005,THE MYSTERY OF THE INVISIBLE THIEVE #8,the-mystery-of-the-invisible-thieve-8,MPHOnline.com,,"Class Code_226, Class_TWEENS (YOUNG READER) (9...",2026-05-17T19:00:00+08:00,None,48302515486877,MYSTERIES8,12.90,0.00,False,Default Title,None,https://mphonline.com/products/the-mystery-of-...
2,8922744717469,THE MYSTERY OF THE PANTOMIME CAT #7,the-mystery-of-the-pantomime-cat-7,MPHOnline.com,,"Class Code_226, Class_TWEENS (YOUNG READER) (9...",2026-05-17T18:59:55+08:00,None,48302515323037,MYSTERIES7,12.90,0.00,False,Default Title,None,https://mphonline.com/products/the-mystery-of-...


In [ ]:
# ── CELL 6 · Operation 3 · Per-collection product lists ─────
"""
OPERATION 3 — Scrape products scoped to each collection
Shopify exposes: /collections/<slug>/products.json
This lets you tag every product with its collection(s).
"""

def scrape_collection_products(collection_slugs: list[str]) -> pd.DataFrame:
    rows = []
    for slug in tqdm(collection_slugs, desc="Collections"):
        page = 1
        while True:
            time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))
            url = f"{BASE_URL}/collections/{slug}/products.json"
            try:
                resp = requests.get(
                    url, headers=HEADERS,
                    params={"limit": 250, "page": page}, timeout=15
                )
                if resp.status_code != 200:
                    break
                products = resp.json().get("products", [])
                if not products:
                    break
                for p in products:
                    rows.append({
                        "collection"  : slug,
                        "product_id"  : p.get("id"),
                        "title"       : p.get("title"),
                        "handle"      : p.get("handle"),
                        "vendor"      : p.get("vendor"),
                        "product_type": p.get("product_type"),
                        "price"       : (p.get("variants") or [{}])[0].get("price"),
                        "available"   : (p.get("variants") or [{}])[0].get("available"),
                        "tags"        : ", ".join(p.get("tags", [])),
                    })
                page += 1
            except Exception as e:
                print(f"  ✗ {slug} page {page}: {e}")
                break

    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_DIR / "collection_products.csv", index=False)
    print(f"✅ Op 3 — {len(df)} collection-product rows → mph_data/collection_products.csv")
    return df

# 'collections' is defined by the preceding "Operation 1" (Cell 4).
# We derive slugs from it to scrape products per collection.
slugs = [c["slug"] for c in collections]
df_coll_products = scrape_collection_products(slugs)  # remove slice for all collections


Collections:   9%|▊         | 31/359 [54:41<21:22:51, 234.67s/it]

In [ ]:
# ── CELL 13 · Operation 10 · Price & availability analysis ──
"""
OPERATION 10 — Price distribution & availability snapshot
Clean, analysis-ready table per product:
  - is_on_sale (bool)
  - discount_amount_myr
  - discount_pct
  - availability status
"""

def build_price_analysis(df_products: pd.DataFrame) -> pd.DataFrame:
    if df_products.empty:
        print("Run Op 2 first.")
        return pd.DataFrame()

    df = df_products.copy()
    df["price_float"]       = pd.to_numeric(df["price_myr"],       errors="coerce")
    df["compare_float"]     = pd.to_numeric(df["compare_at_price"], errors="coerce")
    df["is_on_sale"]        = df["compare_float"] > df["price_float"]
    df["discount_myr"]      = (df["compare_float"] - df["price_float"]).clip(lower=0).round(2)
    df["discount_pct"]      = (
        (df["discount_myr"] / df["compare_float"]) * 100
    ).round(1)
    df["availability"]      = df["available"].map({True: "In Stock", False: "Out of Stock"})

    out = df[[
        "product_id","title","vendor","product_type",
        "price_float","compare_float","is_on_sale",
        "discount_myr","discount_pct","availability","product_url"
    ]].drop_duplicates(subset=["product_id"])

    out.to_csv(OUTPUT_DIR / "price_analysis.csv", index=False)
    print(f"✅ Op 10 — {len(out)} rows → mph_data/price_analysis.csv")

    # Quick stats
    print("\n── Price Stats ──────────────────────────")
    print(out["price_float"].describe().round(2).to_string())
    print(f"\nOn sale : {out['is_on_sale'].sum()} products")
    print(f"In stock: {(out['availability']=='In Stock').sum()} products")
    return out

df_price = build_price_analysis(df_products)


✅ Op 10 — 25000 rows → mph_data/price_analysis.csv

── Price Stats ──────────────────────────
count    25000.00
mean        56.64
std         77.04
min          1.70
25%         24.90
50%         45.90
75%         69.90
max       6430.98

On sale : 0 products
In stock: 8595 products


In [ ]:
import pandas as pd

# ── 1. Load files ──────────────────────────────────────────────────────────────
price_df      = pd.read_csv("mph_data/price_analysis.csv")
products_df   = pd.read_csv("mph_data/products_full.csv")
collection_df = pd.read_csv("mph_data/collection_products.csv")

# ── 2. Normalise column names ──────────────────────────────────────────────────
# price_analysis  →  unified names
price_df = price_df.rename(columns={
    "price_float":    "price_myr",
    "compare_float":  "compare_at_price",
    "availability":   "available_text",   # keep as text; flag column added below
    "discount_myr":   "discount_myr",
    "discount_pct":   "discount_pct",
    "is_on_sale":     "is_on_sale",
})

# collection_products  →  unified names
collection_df = collection_df.rename(columns={
    "price":       "price_myr",
    "collection":  "collection",
})

# ── 3. Add source tag so rows stay traceable ───────────────────────────────────
price_df["source"]      = "price_analysis"
products_df["source"]   = "products_full"
collection_df["source"] = "collection_products"

# ── 4. Combine with pd.concat (outer join fills missing columns with NaN→NULL) ─
combined = pd.concat(
    [price_df, products_df, collection_df],
    ignore_index=True,
    sort=False,          # keep column insertion order
)

# ── 5. Canonical column order ─────────────────────────────────────────────────
preferred_order = [
    "source",
    "product_id",
    "title",
    "handle",
    "vendor",
    "product_type",
    "collection",
    "tags",
    "published_at",
    "description_html",
    "variant_id",
    "sku",
    "variant_title",
    "price_myr",
    "compare_at_price",
    "discount_myr",
    "discount_pct",
    "is_on_sale",
    "available",
    "available_text",
    "image_url",
    "product_url",
]

# Only include columns that actually exist after concat
ordered_cols = [c for c in preferred_order if c in combined.columns]
# Append any extra columns not in the preferred list (future-proofing)
extra_cols   = [c for c in combined.columns if c not in ordered_cols]
combined     = combined[ordered_cols + extra_cols]

# ── 6. Save ────────────────────────────────────────────────────────────────────
output_path = "mph_data/combined.csv"
combined.to_csv(output_path, index=False, na_rep="NULL")

# ── 7. Summary ─────────────────────────────────────────────────────────────────
print(f"Combined rows : {len(combined):,}")
print(f"Columns ({len(combined.columns)}) : {list(combined.columns)}")
print(f"\nRows per source:")
print(combined["source"].value_counts().to_string())
print(f"\nSaved → {output_path}")

Combined rows : 546,650
Columns (22) : ['source', 'product_id', 'title', 'handle', 'vendor', 'product_type', 'collection', 'tags', 'published_at', 'description_html', 'variant_id', 'sku', 'variant_title', 'price_myr', 'compare_at_price', 'discount_myr', 'discount_pct', 'is_on_sale', 'available', 'available_text', 'image_url', 'product_url']

Rows per source:
source
collection_products    496650
price_analysis          25000
products_full           25000

Saved → mph_data/combined.csv
